# 環境建置與圖片載入

In [ ]:
!pip install opencv-python
!pip install numpy

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

print(f"OpenCv 版本：{cv2.__version__}")

In [ ]:
from google.colab import files
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from google.colab.patches import cv2_imshow
import io

In [ ]:
uploaded = files.upload()

img = None

for filename in uploaded.keys():
  img = cv2.imread(filename)
  image = Image.open(io.BytesIO(uploaded[filename]))
  plt.figure(figsize=(8, 6))
  plt.imshow(image)
  plt.title(filename)
  plt.axis('off')
  plt.show()
  if img is not None:
    print(f"成功讀取圖片： {filename}")
    break

if img is None:
  raise ValueError("Error: 圖片無法讀取，請確認路徑或格式是否正確")

# 復古 CV 風格
1. 實驗一：對比 sobel, scharr, laplacian, canny
2. 實驗二：對比二值化固定閾值, Otsu 全域自適應, Otsu 局部自適應
3. 實驗三：對比 Otsu + 不同形態學
4. 最終渲染

## 實驗一效果對比

### EdgeDetector
EdgeDetector 後面可以直接迭代比較，生成不同邊緣偵測方式的圖片
1. get_sobel 實作 sobel
2. get_scharr 實作 scharr
3. get_laplacian 實作 laplacian
4. get_canny 實作 canny
4. get_all_edges 留作後續自動生成大量比較的基礎

In [ ]:
class EdgeDetector:
    def __init__(self, img_array):
        self.gray = cv2.cvtColor(img_array, cv2.COLOR_BGR2GRAY)
        self.blurred = cv2.GaussianBlur(self.gray, (3, 3), 0)

    def get_sobel(self, ksize=3):
        grad_x = cv2.Sobel(self.blurred, cv2.CV_64F, 1, 0, ksize=ksize)
        grad_y = cv2.Sobel(self.blurred, cv2.CV_64F, 0, 1, ksize=ksize)
        return cv2.addWeighted(cv2.convertScaleAbs(grad_x), 0.5, cv2.convertScaleAbs(grad_y), 0.5, 0)

    def get_scharr(self):
        grad_x = cv2.Scharr(self.blurred, cv2.CV_64F, 1, 0)
        grad_y = cv2.Scharr(self.blurred, cv2.CV_64F, 0, 1)
        return cv2.addWeighted(cv2.convertScaleAbs(grad_x), 0.5, cv2.convertScaleAbs(grad_y), 0.5, 0)

    def get_laplacian(self, ksize=3):
        return cv2.convertScaleAbs(cv2.Laplacian(self.blurred, cv2.CV_64F, ksize=ksize))

    def get_canny(self, th1=50, th2=150):
        return cv2.Canny(self.blurred, th1, th2)

    def get_all_edges(self):
        return {
            'Sobel': self.get_sobel(),
            'Scharr': self.get_scharr(),
            'Laplacian': self.get_laplacian(),
            'Canny': self.get_canny()
        }


### 主程式
橫向對比四種邊緣偵測的效果
1. detector 將圖片初始化
2. edge_results 取得方法數的 dictionary
3. 迭代各方法的效果並加入畫布

In [ ]:
detector = EdgeDetector(img)

edge_results = detector.get_all_edges()

plt.figure(figsize=(20, 5))  # 平行對比

for i, (name, edge_img) in enumerate(edge_results.items()):
    plt.subplot(1, 4, i+1)
    plt.imshow(edge_img, cmap='gray')
    plt.title(f"{name} Edge")
    plt.axis('off')

plt.tight_layout()
plt.show()

## 實驗二效果對比

### BinarizationExperiment
BinarizationExperiment 後面可以直接迭代比較，生成不同二值化的圖片
1. get_fixed 實作邊緣固定 (50)
2. get_otsu 實作 Otsu 全域自適應
3. get_adaptive 實作 Otsu 局部自適應 (block_size = 11, C = -2)
4. get_all_binarizations 留作後續自動生成大量比較的基礎

In [ ]:
class BinarizationExperiment:
    def __init__(self, raw_edge_img):
        self.raw_edge = raw_edge_img

    def get_fixed(self, thresh=50):
        _, binary = cv2.threshold(self.raw_edge, thresh, 255, cv2.THRESH_BINARY)
        return binary

    def get_otsu(self):
        ret_val, binary = cv2.threshold(self.raw_edge, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return binary, ret_val

    def get_adaptive(self, block_size=11, C=-2):
        binary = cv2.adaptiveThreshold(self.raw_edge, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, block_size, C)
        return binary

    def get_all_binarizations(self):
        fixed_bin = self.get_fixed(50)
        otsu_bin, otsu_val = self.get_otsu()
        adaptive_bin = self.get_adaptive()

        return {
            'Fixed': fixed_bin,
            f'Otsu (Auto Th={int(otsu_val)})': otsu_bin,
            'Local Adaptive': adaptive_bin
        }

### 主程式
橫向對比三種二值化 + 四種邊緣偵測的效果
1. edge_results 取得邊緣方法數的 dictionary
2. bin_results 取得二值化方法數的 dictionary
3. 迭代各方法的效果並加入畫布

In [ ]:
edge_results = detector.get_all_edges()

# 4 列 (邊緣算法) x 3 行 (二值化) 的畫布
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(18, 16))

for i, (edge_name, raw_edge) in enumerate(edge_results.items()):
    # 將每一種邊緣結果丟入二值化實驗模組
    binarizer = BinarizationExperiment(raw_edge)
    bin_results = binarizer.get_all_binarizations()

    for j, (bin_name, bin_img) in enumerate(bin_results.items()):
        ax = axes[i, j]
        ax.imshow(bin_img, cmap='gray')

        # 只在第一列顯示二值化策略的 Title
        if i == 0:
            ax.set_title(bin_name, fontsize=14, pad=10)

        # 只在第一行顯示邊緣算法的 Y 軸標籤
        if j == 0:
            ax.set_ylabel(edge_name, fontsize=14, rotation=90, labelpad=20)

        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()

## 實驗三效果對比

### MorphologyExperiment
MorphologyExperiment 後面可以直接迭代比較，生成不同型態學的圖片
1. get_dilated 實作 dilated
2. get_eroded 實作 eroded
3. get_opened 實作 MORPH_OPEN
4. get_closed 實作 MORPH_CLOSE
5. get_none 實作 跳過形態學
5. get_all_morphologies 留作後續自動生成大量比較的基礎

In [ ]:
class MorphologyExperiment:
    def __init__(self, binary_img):
        self.binary_base = binary_img
        # 定義 3x3 的矩形結構元素 (Kernel)
        self.kernel = np.ones((3, 3), np.uint8)

    def get_dilated(self, iterations=1):
        return cv2.dilate(self.binary_base, self.kernel, iterations=iterations)

    def get_eroded(self, iterations=1):
        return cv2.erode(self.binary_base, self.kernel, iterations=iterations)

    def get_opened(self):
        return cv2.morphologyEx(self.binary_base, cv2.MORPH_OPEN, self.kernel)

    def get_closed(self):
        return cv2.morphologyEx(self.binary_base, cv2.MORPH_CLOSE, self.kernel)

    def get_none(self):
        return self.binary_base

    def get_all_morphologies(self):
        return {
            'Dilation': self.get_dilated(),
            'Erosion': self.get_eroded(),
            'Opening': self.get_opened(),
            'Closing': self.get_closed(),
            'None': self.get_none()
        }

### 主程式
橫向對比四種型態學 + 三種二值化 + 四種邊緣偵測的效果
1. edge_results 取得邊緣方法數的 dictionary
2. bin_results 取得二值化方法數的 dictionary
3. morph_results 取得形態學方法樹的 dictionay
4. 迭代各方法的效果並加入畫布

In [ ]:
detector = EdgeDetector(img)
edge_results = detector.get_all_edges()

# 定義 12 列 (4 邊緣 * 3 二值化) x 5 行 (形態學) 的畫布
fig, axes = plt.subplots(nrows=12, ncols=5, figsize=(24, 75))

edge_names = list(edge_results.keys())

for i, (edge_name, raw_edge) in enumerate(edge_results.items()):
    # 二值化 3 種方法
    binarizer = BinarizationExperiment(raw_edge)
    bin_results = binarizer.get_all_binarizations()
    bin_names = list(bin_results.keys())

    for j, (bin_name, bin_img) in enumerate(bin_results.items()):
        current_row = i * 3 + j

        # 形態學 5 種方法
        morpher = MorphologyExperiment(bin_img)
        morph_results = morpher.get_all_morphologies()
        morph_names = list(morph_results.keys())

        for k, (morph_name, morph_img) in enumerate(morph_results.items()):
            ax = axes[current_row, k]
            ax.imshow(morph_img, cmap='gray')

            if current_row == 0:
                ax.set_title(f"Morph: {morph_name}", fontsize=18, pad=20)

            if k == 0:
                y_label = f"{edge_name}\n{bin_name}"
                ax.set_ylabel(y_label, fontsize=16, rotation=0, labelpad=80, va='center')

            ax.set_xticks([])
            ax.set_yticks([])

plt.tight_layout()
plt.show()

## 最終渲染


### RetroCRTRenderer

In [ ]:
class RetroCRTRenderer:
    def __init__(self, scanline_step=3, glow_strength=0.7):
        self.scanline_step = scanline_step
        self.glow_strength = glow_strength

    def render(self, binary_mask):
        h, w = binary_mask.shape
        # 色彩映射 (BGR: 綠色)
        crt_img = np.zeros((h, w, 3), dtype=np.uint8)
        crt_img[binary_mask > 0] = [0, 255, 0]

        # 模擬磷光溢色
        glow = cv2.GaussianBlur(crt_img, (5, 5), 0)
        crt_img = cv2.addWeighted(crt_img, 1.0, glow, 0.7, 0)

        # 模擬掃描線
        crt_img[::self.scanline_step, :] = (crt_img[::self.scanline_step, :] * 0.4).astype(np.uint8)

        # 模擬雜訊
        noise = np.zeros_like(crt_img)
        cv2.randu(noise, 0, 50)
        return cv2.add(crt_img, noise)

### 主程式
橫向對比四種型態學 + 三種二值化 + 四種邊緣偵測後著色為 **復古CV** 的效果
1. edge_results 取得邊緣方法數的 dictionary
2. bin_results 取得二值化方法數的 dictionary
3. morph_results 取得形態學方法樹的 dictionay
4. 迭代各方法的效果並加入畫布

In [ ]:
detector = EdgeDetector(img)
edge_results = detector.get_all_edges()

# 定義 12 列 (4 邊緣 * 3 二值化) x 5 行 (形態學) 的畫布
fig, axes = plt.subplots(nrows=12, ncols=5, figsize=(24, 75))

renderer = RetroCRTRenderer()

for i, (edge_name, raw_edge) in enumerate(edge_results.items()):

    # 二值化 3 種方法
    binarizer = BinarizationExperiment(raw_edge)
    bin_results = binarizer.get_all_binarizations()

    for j, (bin_name, bin_img) in enumerate(bin_results.items()):
        current_row = i * 3 + j

        # 形態學 5 種方法
        morpher = MorphologyExperiment(bin_img)
        morph_results = morpher.get_all_morphologies()

        for k, (morph_name, morph_img) in enumerate(morph_results.items()):
            # 在實例上呼叫 render 方法
            final_rendered = renderer.render(morph_img)

            ax = axes[current_row, k]
            # OpenCV BGR -> Matplotlib RGB
            ax.imshow(cv2.cvtColor(final_rendered, cv2.COLOR_BGR2RGB))

            if current_row == 0:
                ax.set_title(f"Morph: {morph_name}", fontsize=18, pad=20, fontweight='bold')

            if k == 0:
                y_label = f"{edge_name}\n{bin_name}"
                ax.set_ylabel(y_label, fontsize=16, rotation=0, labelpad=90, va='center', fontweight='bold')

            ax.set_xticks([])
            ax.set_yticks([])

plt.tight_layout()
plt.show()

# 水彩風格
1. 實驗一：對比 高斯模糊、中值模糊、雙邊濾波
2. 實驗二：對比 laplacian, canny, sketch-divide
3. 實驗三：對比 multiply, alpha blending, soft light，並做最終渲染

## 實驗一效果對比

### WatercolorSimplifier
1. get_gaussian 高斯模糊
2. get_median 中值模糊
3. get_bilateral 雙邊濾波
4. get_all_simplifications 留作後續自動生成大量比較的基礎

In [ ]:
class WatercolorSimplifier:
    def __init__(self, img_array):
        self.img = img_array

    def get_gaussian(self, ksize=15):
        return cv2.GaussianBlur(self.img, (ksize, ksize), 0)

    def get_median(self, ksize=15):
        return cv2.medianBlur(self.img, ksize)

    def get_bilateral(self, d=9, sigma_color=75, sigma_space=75, iterations=3):
        temp = self.img.copy()
        for _ in range(iterations):
            temp = cv2.bilateralFilter(temp, d, sigma_color, sigma_space)
        return temp

    def get_all_simplifications(self):
        return {
            '1. Gaussian (k=15)': self.get_gaussian(),
            '2. Median (k=15)': self.get_median(),
            '3. Bilateral (Iter=3)': self.get_bilateral(iterations=3),
            '4. Bilateral (Iter=7)': self.get_bilateral(iterations=7)  # 多比一個深度
        }

### 主程式

In [ ]:
simplifier = WatercolorSimplifier(img)
smooth_results = simplifier.get_all_simplifications()

plt.figure(figsize=(24, 6))
for i, (name, s_img) in enumerate(smooth_results.items()):
    plt.subplot(1, 4, i+1)
    # OpenCV BGR -> RGB
    plt.imshow(cv2.cvtColor(s_img, cv2.COLOR_BGR2RGB))
    plt.title(name, fontsize=16, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()

## 實驗二效果對比

### WatercolorSketcher
1. get_canny 實作 canny
2. get_laplacian 實作 laplacian
3. get_sketch_divide 實作 sketch-divide
4. get_all_sketches 留作後續自動生成大量比較的基礎

In [ ]:
class WatercolorSketcher:
    def __init__(self, img_array):
        self.gray = cv2.cvtColor(img_array, cv2.COLOR_BGR2GRAY)
        self.blurred_base = cv2.GaussianBlur(self.gray, (3, 3), 0)

    def get_canny(self, th1=50, th2=150):
        edge = cv2.Canny(self.blurred_base, th1, th2)
        return cv2.bitwise_not(edge)

    def get_laplacian(self, ksize=3):
        lap = cv2.Laplacian(self.blurred_base, cv2.CV_64F, ksize=ksize)
        lap_abs = cv2.convertScaleAbs(lap)
        return cv2.bitwise_not(lap_abs)

    def get_sketch_divide(self, ksize=21):
        inverted_gray = 255 - self.gray
        inverted_blur = cv2.GaussianBlur(inverted_gray, (ksize, ksize), 0)
        sketch = cv2.divide(self.gray, 255 - inverted_blur, scale=256)
        return sketch

    def get_all_sketches(self):
        return {
            '1. Canny (Binary)': self.get_canny(),
            '2. Laplacian (Gradient)': self.get_laplacian(),
            '3. Sketch-Divide (Soft)': self.get_sketch_divide(ksize=21),
            '4. Sketch-Divide (Hard)': self.get_sketch_divide(ksize=7)
        }

### 主程式

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 實驗一 4 種方法
simplifier = WatercolorSimplifier(img)
smooth_results = simplifier.get_all_simplifications()
s_names = list(smooth_results.keys())

# 實驗二 3 種方法
sketcher = WatercolorSketcher(img)
sketch_results = {
    'Laplacian': sketcher.get_laplacian(),
    'Canny': sketcher.get_canny(),
    'Sketch-Divide': sketcher.get_sketch_divide()
}
k_names = list(sketch_results.keys())

# 定義畫布：4 列 (平滑方法) x 3 行 (素描方法)
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(18, 24))

for i, s_name in enumerate(s_names):
    s_img = smooth_results[s_name] # 彩色色塊層

    for j, k_name in enumerate(k_names):
        k_img = sketch_results[k_name] # 灰階素描層

        k_bgr = cv2.cvtColor(k_img, cv2.COLOR_GRAY2BGR)

        combined = cv2.multiply(s_img, k_bgr, scale=1/255)

        ax = axes[i, j]
        ax.imshow(cv2.cvtColor(combined, cv2.COLOR_BGR2RGB))

        if i == 0:
            ax.set_title(f"Outline: {k_name}", fontsize=16, pad=15, fontweight='bold')

        if j == 0:
            ax.set_ylabel(f"Base: {s_name}", fontsize=14, rotation=90, labelpad=20, fontweight='bold')

        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()

## 實驗三效果對比 & 最終渲染

### WatercolorRenderer
1. blend_multiply 實作 multiply
2. blend_alpha 實作 alpha blending
3. blend_soft_light 實作 soft light
4. 留作後續自動生成大量比較的基礎

In [ ]:
class WatercolorRenderer:
    def __init__(self, color_img, sketch_img):
        self.color = color_img

        if len(sketch_img.shape) == 2:
            self.sketch = cv2.cvtColor(sketch_img, cv2.COLOR_GRAY2BGR)
        else:
            self.sketch = sketch_img

    def blend_multiply(self):
        return cv2.multiply(self.color, self.sketch, scale=1/255)

    def blend_alpha(self, alpha=0.3):
        return cv2.addWeighted(self.color, 1-alpha, self.sketch, alpha, 0)

    def blend_soft_light(self):
        # 歸一化到 0-1 進行數學運算
        img1 = self.color.astype(float) / 255.0
        img2 = self.sketch.astype(float) / 255.0

        # Soft Light 公式
        result = (1 - 2 * img2) * (img1 ** 2) + 2 * img2 * img1
        return (result * 255).astype(np.uint8)

    def get_all_blends(self):
        return {
            'Multiply': self.blend_multiply(),
            'Alpha (0.3)': self.blend_alpha(),
            'Soft Light': self.blend_soft_light()
        }

### 主程式

In [ ]:
simplifier = WatercolorSimplifier(img)
smooth_results = simplifier.get_all_simplifications()
s_names = list(smooth_results.keys())

sketcher = WatercolorSketcher(img)
sketch_results = {
    'Canny': sketcher.get_canny(),
    'Laplacian': sketcher.get_laplacian(),
    'Sketch-Divide': sketcher.get_sketch_divide()
}
k_names = list(sketch_results.keys())

# 建立畫布：12 列 x 3 行
fig, axes = plt.subplots(nrows=12, ncols=3, figsize=(18, 72))

for i, s_name in enumerate(s_names):
    s_img = smooth_results[s_name]

    for j, k_name in enumerate(k_names):
        current_row = i * 3 + j
        k_img = sketch_results[k_name]

        renderer = WatercolorRenderer(s_img, k_img)
        blend_results = renderer.get_all_blends()
        b_names = list(blend_results.keys())

        for k, b_name in enumerate(b_names):
            final_img = blend_results[b_name]

            ax = axes[current_row, k]
            ax.imshow(cv2.cvtColor(final_img, cv2.COLOR_BGR2RGB))

            if current_row == 0:
                ax.set_title(f"Blend: {b_name}", fontsize=18, pad=20, fontweight='bold')

            if k == 0:
                y_label = f"Smooth:\n{s_name}\nSketch: {k_name}"
                ax.set_ylabel(y_label, fontsize=14, rotation=0,
                             labelpad=110, va='center', ha='right', fontweight='bold')

            ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.subplots_adjust(left=0.25)
plt.show()

# 油畫風格
實驗：對比 不同畫筆尺寸、不同平滑比例，並做最終渲染

## 實驗效果對比 & 最終渲染

### OilPaintingExperiment

In [ ]:
class OilPaintingExperiment:
    def __init__(self, img_array):
        self.img = img_array

    def get_oil_painting(self, size=7, dynRatio=1):
        try:
            return cv2.xphoto.oilPainting(self.img, size, dynRatio)
        except cv2.error:
            print("油畫效果需要 Opencv contrib 模組，使用 pip install opencv-contrib-python 安裝")
            return self.img

### 主程式

In [ ]:
oil_exp = OilPaintingExperiment(img)

sizes = [3, 5, 7, 11] # 筆畫大小
dyn_ratios = [1, 4, 7, 10]  # 平滑程度

# 建立 4 x 4 畫布
fig, axes = plt.subplots(nrows=len(sizes), ncols=len(dyn_ratios), figsize=(20, 20))

for i, current_size in enumerate(sizes):
    for j, current_ratio in enumerate(dyn_ratios):
        oil_img = oil_exp.get_oil_painting(size=current_size, dynRatio=current_ratio)

        ax = axes[i, j]
        # OpenCV BGR -> Matplotlib RGB
        ax.imshow(cv2.cvtColor(oil_img, cv2.COLOR_BGR2RGB))

        if i == 0:
            ax.set_title(f"dynRatio: {current_ratio}", fontsize=18, pad=15, fontweight='bold')

        if j == 0:
            ax.set_ylabel(f"size: {current_size}", fontsize=18, rotation=0,
                          labelpad=80, va='center', ha='right', fontweight='bold')

        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.subplots_adjust(left=0.15)
plt.show()

# 素描風格
1. 實驗一：對比 sobel, scharr, laplacian, canny, sketch-divide (筆畫提取)
2. 實驗二：對比 normal, gamma, clahe (模擬陰影) & 最終渲染

## 實驗一效果對比

### SketchExtractor
1. get_sobel 實作 sobel
2. get_scharr 實作 scharr
3. get_laplacian 實作 laplacian
4. get_canny 實作 canny
5. get_sketch_divide 實作 sketch-divide
6. get_all_sketches 留作後續自動生成大量比較的基礎

In [ ]:
class SketchExtractor:
    def __init__(self, img_array):
        self.gray = cv2.cvtColor(img_array, cv2.COLOR_BGR2GRAY)
        self.blurred_base = cv2.GaussianBlur(self.gray, (3, 3), 0)

    def get_sobel(self, ksize=3):
        grad_x = cv2.Sobel(self.blurred_base, cv2.CV_64F, 1, 0, ksize=ksize)
        grad_y = cv2.Sobel(self.blurred_base, cv2.CV_64F, 0, 1, ksize=ksize)
        sobel = cv2.addWeighted(cv2.convertScaleAbs(grad_x), 0.5, cv2.convertScaleAbs(grad_y), 0.5, 0)
        return cv2.bitwise_not(sobel)

    def get_scharr(self):
        grad_x = cv2.Scharr(self.blurred_base, cv2.CV_64F, 1, 0)
        grad_y = cv2.Scharr(self.blurred_base, cv2.CV_64F, 0, 1)
        scharr = cv2.addWeighted(cv2.convertScaleAbs(grad_x), 0.5, cv2.convertScaleAbs(grad_y), 0.5, 0)
        return cv2.bitwise_not(scharr)

    def get_laplacian(self, ksize=3):
        lap = cv2.Laplacian(self.blurred_base, cv2.CV_64F, ksize=ksize)
        return cv2.bitwise_not(cv2.convertScaleAbs(lap))

    def get_canny(self, th1=50, th2=150):
        edge = cv2.Canny(self.blurred_base, th1, th2)
        return cv2.bitwise_not(edge)

    def get_sketch_divide(self, ksize=21):
        inverted_gray = 255 - self.gray
        inverted_blur = cv2.GaussianBlur(inverted_gray, (ksize, ksize), 0)
        return cv2.divide(self.gray, 255 - inverted_blur, scale=256)

    def get_all_sketches(self):
        return {
            'Sobel': self.get_sobel(),
            'Scharr': self.get_scharr(),
            'Laplacian': self.get_laplacian(),
            'Canny': self.get_canny(),
            'Sketch-Divide': self.get_sketch_divide(ksize=21)
        }

### 主程式

In [ ]:
extractor = SketchExtractor(img)
sketch_results = extractor.get_all_sketches()

# 1 x 5 的畫布
plt.figure(figsize=(25, 6))

for i, (name, s_img) in enumerate(sketch_results.items()):
    plt.subplot(1, 5, i+1)

    plt.imshow(s_img, cmap='gray')
    plt.title(f"{name}", fontsize=18, fontweight='bold', pad=15)
    plt.axis('off')

plt.tight_layout()
plt.show()

## 實驗二效果對比

### ShadingExperiment
1. get_normal 實作 normal
2. get_gamma 實作 gamma
3. get_clahe 實作 clahe
4. get_all_shadings 留作後續自動生成大量比較的基礎

In [ ]:
class ShadingExperiment:
    def __init__(self, sketch_img):
        self.sketch = sketch_img.astype(np.uint8)

    def get_normal(self):
        return self.sketch

    def get_gamma(self, gamma=0.75):
        invGamma = 1.0 / gamma
        table = np.array([((i / 255.0) ** invGamma) * 255
                          for i in np.arange(0, 256)]).astype("uint8")
        return cv2.LUT(self.sketch, table)

    def get_clahe(self, clip_limit=2.0, tile_grid=(8, 8)):
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
        return clahe.apply(self.sketch)

    def get_all_shadings(self):
        return {
            'Normal': self.get_normal(),
            'Gamma': self.get_gamma(),
            'CLAHE': self.get_clahe()
        }

### 主程式

In [ ]:
k_names = list(sketch_results.keys())

# 建立 5 列 x 3 行的畫布
fig, axes = plt.subplots(nrows=len(k_names), ncols=3, figsize=(15, 25))

for i, k_name in enumerate(k_names):
    k_img = sketch_results[k_name]

    shader = ShadingExperiment(k_img)
    shade_results = shader.get_all_shadings()
    sh_names = list(shade_results.keys())

    for j, sh_name in enumerate(sh_names):
        sh_img = shade_results[sh_name]

        ax = axes[i, j]
        ax.imshow(sh_img, cmap='gray', vmin=0, vmax=255)

        if i == 0:
            ax.set_title(f"Shading: {sh_name}", fontsize=18, pad=15, fontweight='bold')

        if j == 0:
            ax.set_ylabel(f"Extract:\n{k_name}", fontsize=16, rotation=0,
                          labelpad=80, va='center', ha='right', fontweight='bold')

        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.subplots_adjust(left=0.25)
plt.show()

# 最終結果圖
* 復古CV：Sobel+Fixed
* 水彩：雙邊濾波+sketch-divide+multiply
* 油畫：畫筆尺寸11、平滑比例1
* 素描：Sobel+Gamma

In [ ]:
# 復古CV
retro_detector = EdgeDetector(img)
sobel_edge = retro_detector.get_sobel()

retro_binarizer = BinarizationExperiment(sobel_edge)
retro_mask, _ = retro_binarizer.get_otsu()

retro_morpher = MorphologyExperiment(retro_mask)
final_mask = retro_morpher.get_none()

retro_renderer = RetroCRTRenderer()
final_retro = retro_renderer.render(final_mask)

# 水彩
wc_simplifier = WatercolorSimplifier(img)
color_layer = wc_simplifier.get_bilateral(iterations=3)

wc_sketcher = WatercolorSketcher(img)
sketch_layer = wc_sketcher.get_sketch_divide()

wc_renderer = WatercolorRenderer(color_layer, sketch_layer)
final_watercolor = wc_renderer.blend_multiply()

# 油畫
oil_exp = OilPaintingExperiment(img)
final_oil = oil_exp.get_oil_painting(size=3, dynRatio=1)

# 素描
sk_extractor = SketchExtractor(img)
sobel_sketch_base = sk_extractor.get_sobel()

sk_shader = ShadingExperiment(sobel_sketch_base)
final_sketch = sk_shader.get_gamma(gamma=0.75)

cv2.imwrite('output_retro.jpg', final_retro)
cv2.imwrite('output_watercolor.jpg', final_watercolor)
cv2.imwrite('output_oil.jpg', final_oil)
cv2.imwrite('output_sketch.jpg', final_sketch)